# Relabel the dataset: the actions taken while asleep are noop

Craftax replaces every action by a noop while the player sleeps (`craftax_step` starts with `action = select(state.is_sleeping, NOOP, action)`), so the policy's choice in those steps (about 17% of them) had no effect. The files on the Hub hold the policy's choice in `action`. This notebook downloads them, keeps the policy's choice in a new column `action_policy`, writes the executed actions into `action` (`purejaxql/scripts/relabel_asleep.py`, the collector's own rule), checks the result, and replaces the files on the Hub. Nothing else in the files changes.

| step | what |
|---|---|
| 0 | configuration, Hugging Face login, a CPU JAX environment |
| 1 | download the three files from the Hub (the revision is recorded) |
| 2 | **relabel** in place |
| 3 | **checks** on the CPU: every file replays from `seed` + the new `action` (all columns identical) and `action` follows the rule; the dataset numbers (D0) from the relabeled files |
| 4 | **replace** the files on the Hub (only when every check passed), and the reports |

The old files stay in the history of the Hub repo (the revision printed in step 1). Runs on a CPU pod or any machine with about 20 GB of disk; the replay check takes a few minutes per file.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
from pathlib import Path

VOLUME = Path("/workspace")
REPO = VOLUME / "AAC"                        # the git clone, at the commit with relabel_asleep.py
VENV = VOLUME / "venvs" / "jax-cpu"          # CPU JAX: the relabeling and the replay check need no GPU
WORK = VOLUME / "relabel"                    # the downloaded files are relabeled here (about 10 GB)

HF_REPO_ID = "your-user/craftax-classic"     # the dataset repo of runpod_dataset.ipynb (data/, reports/)
SPLITS = ("train", "val", "test")
REPLAY_EPISODES = 20                         # episodes per file replayed in step 3

In [ ]:
# --- 0. Helpers, Hugging Face login, the CPU JAX environment ---------------------------------------------------------
import getpass, hashlib, json, os, subprocess, sys

os.chdir(VOLUME)
PYTHON = VENV / "bin" / "python"
DATA = WORK / "data"                         # hf_hub_download(local_dir=WORK) puts data/<file> here
ENV = {
    **{k: v for k, v in os.environ.items() if k != "LD_LIBRARY_PATH"},
    "JAX_PLATFORMS": "cpu",                  # the planning environment replays on the CPU: check there
    "HDF5_USE_FILE_LOCKING": "FALSE",
    "PYTHONUNBUFFERED": "1",
}


def sh(cmd, cwd=None, check=True):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(c[:100] for c in cmd))
    code = subprocess.call(cmd, env=ENV, cwd=cwd)
    if check and code:
        raise RuntimeError(f"{cmd[0]} exited with {code}")
    return code


sh([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=2"])
from huggingface_hub import HfApi, hf_hub_download

api = HfApi(token=os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (write access): "))
print("logged in as", api.whoami()["name"])

assert (REPO / "purejaxql" / "scripts" / "relabel_asleep.py").exists(), "pull the repository: relabel_asleep.py is missing"
sh(["git", "-C", REPO, "log", "-1", "--oneline"])
if not PYTHON.exists():
    sh([sys.executable, "-m", "venv", VENV])
    sh([PYTHON, "-m", "pip", "install", "-q", "-U", "pip"])
    sh([PYTHON, "-m", "pip", "install", "-q", "-r", "environment/jax-requirements.txt", "matplotlib"], cwd=REPO)
sh([PYTHON, "-c", "import jax, craftax, h5py; print('jax', jax.__version__, jax.devices())"])

## 1. Download

In [ ]:
# --- 1. The files from the Hub, and the revision they come from --------------------------------------------------------
WORK.mkdir(parents=True, exist_ok=True)
SOURCE_REVISION = api.list_repo_commits(HF_REPO_ID, repo_type="dataset")[0].commit_id
print("the files before the relabeling are at revision", SOURCE_REVISION)
(WORK / "source_revision.txt").write_text(SOURCE_REVISION + "\n")
for split in SPLITS:
    path = DATA / f"craftax_expert_{split}.h5"
    if not path.exists():
        hf_hub_download(HF_REPO_ID, f"data/{path.name}", repo_type="dataset", revision=SOURCE_REVISION, local_dir=WORK,
                        token=api.token)
    print(split, f"{path.stat().st_size / 2**30:.2f} GiB")

## 2. Relabel

In place: `action_policy` is written first (the policy's choice, never overwritten), then `action`. An interrupted run can be started again; running it again gives the same files.

In [ ]:
# --- 2. Relabel ------------------------------------------------------------------------------------------------------------
files = [DATA / f"craftax_expert_{split}.h5" for split in SPLITS]
sh([PYTHON, REPO / "purejaxql" / "scripts" / "relabel_asleep.py", *[x for p in files for x in ("--data", p)]])
RECORDS = {}
for split, path in zip(SPLITS, files):
    out = subprocess.run([PYTHON, "-c", "import h5py, sys; print(h5py.File(sys.argv[1], 'r').attrs['relabel'])", path],
                         env=ENV, capture_output=True, text=True, check=True)
    RECORDS[split] = json.loads(out.stdout)
    r = RECORDS[split]
    print(f"{split}: {r['asleep_steps']:,} of {r['steps']:,} steps taken asleep, {r['changed']:,} relabeled ({100 * r['changed_share']:.1f}%)")

## 3. Checks

**Replay:** a sample of every file is re-simulated from its seed and the new `action` column; every discrete column must be identical (the game ignored the actions that changed), and `action` must follow the rule. **Dataset numbers:** `dataset_section compute` on the relabeled files; the action coverage now counts a step taken asleep as a noop, and its `asleep_steps` must equal the relabeling's count.

In [ ]:
# --- 3. Replay check on the CPU and the dataset numbers ---------------------------------------------------------------------
CHECKS = {}
for split, path in zip(SPLITS, files):
    print(f"===== replay {split}")
    CHECKS[f"replay_{split}"] = sh([PYTHON, REPO / "purejaxql" / "scripts" / "check_replay.py", "--data", path,
                                    "--num-episodes", REPLAY_EPISODES], cwd=REPO / "purejaxql", check=False) == 0

REPORTS = WORK / "reports" / "dataset"
sh([PYTHON, "-m", "dataset_section", "compute", *[x for s, p in zip(SPLITS, files) for x in ("--data", p, "--label", s)],
    "--out", REPORTS, "--sha256"], cwd=REPO / "analysis")
sh([PYTHON, "-m", "dataset_section", "plot", "--out", REPORTS], cwd=REPO / "analysis")
numbers = json.loads((REPORTS / "dataset_numbers.json").read_text())
for split in SPLITS:
    asleep = numbers["splits"][split]["actions"]["asleep_steps"]
    CHECKS[f"asleep_{split}"] = asleep == RECORDS[split]["asleep_steps"]
    print(f"{split}: dataset numbers count {asleep:,} asleep steps, the relabeling {RECORDS[split]['asleep_steps']:,}")
print(CHECKS)
assert all(CHECKS.values()), "a check failed: nothing is uploaded"

## 4. Replace the files on the Hub

Only after every check passed. The upload replaces `data/craftax_expert_*.h5` and the dataset reports in one commit each; the files before the relabeling stay at the revision of step 1.

In [ ]:
# --- 4. Upload (only when every check passed) -----------------------------------------------------------------------------
assert CHECKS and all(CHECKS.values()), "run the checks of step 3 first"
message = f"Relabel the actions taken while asleep as noop (action_policy keeps the policy's choice); before: {SOURCE_REVISION}"
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA), path_in_repo="data",
                  allow_patterns=["craftax_expert_*.h5"], commit_message=message)
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(REPORTS), path_in_repo="reports/dataset",
                  commit_message="Dataset numbers of the relabeled files")
print(f"done: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/data (the old files: revision {SOURCE_REVISION})")
print("copy reports/dataset into the repository (reports/dataset/) and commit it, so the hashes there match the Hub")